# 01 · Exploratory Data Analysis — Hyperliquid Trader Data
> **Goal:** Understand the shape, quality, and distribution of the raw trader dataset.

In [1]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is available")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU not available")



✅ GPU is available
GPU Name: NVIDIA GeForce GTX 1650


In [2]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from data_loader import load_trades
from preprocessing import clean_trades

%matplotlib inline
sns.set_theme(style="darkgrid")


## 1. Load Raw Data

In [3]:
trades_raw = load_trades()
trades_raw.head()


FileNotFoundError: Trades file not found at: D:\Documents\norse\web Applicarion\Prime Trade\data\raw\historical_trades.csv
Download it from the Google Drive link and place it in data/raw/.

## 2. Schema & Data Types

In [ ]:
print(trades_raw.dtypes)
print(f"\nShape: {trades_raw.shape}")


## 3. Missing Values

In [ ]:
null_pct = trades_raw.isnull().mean().sort_values(ascending=False) * 100
null_pct[null_pct > 0].plot(kind="bar", title="% Missing per Column", figsize=(12, 4))
plt.ylabel("% Null")
plt.tight_layout()
plt.show()
print(null_pct[null_pct > 0].to_string())


## 4. Clean & Basic Stats

In [ ]:
trades = clean_trades(trades_raw)
trades.describe()


## 5. PnL Distribution

In [ ]:
fig = px.histogram(
    trades, x="closedpnl", nbins=100,
    title="Distribution of Closed PnL",
    color_discrete_sequence=["#3B82F6"],
    template="plotly_dark",
)
fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="Break-even")
fig.show()


## 6. Trades by Symbol

In [ ]:
sym_counts = trades["symbol"].value_counts().head(20)
px.bar(sym_counts, title="Top 20 Traded Symbols", template="plotly_dark",
       labels={"value": "Trade Count", "index": "Symbol"}).show()


## 7. Trade Volume Over Time

In [ ]:
daily = trades.groupby("trade_date").size().reset_index(name="trade_count")
px.line(daily, x="trade_date", y="trade_count",
        title="Daily Trade Count Over Time", template="plotly_dark").show()


## 8. Side Distribution (Long vs. Short)

In [ ]:
if "side" in trades.columns:
    px.pie(trades, names="side", title="Long vs Short Trade Distribution",
           template="plotly_dark").show()


## 9. Leverage Distribution

In [ ]:
if "leverage" in trades.columns:
    px.histogram(trades, x="leverage", nbins=50,
                 title="Leverage Distribution", template="plotly_dark").show()


## 10. Top 20 Accounts by Trade Count

In [ ]:
top_traders = trades["account"].value_counts().head(20).reset_index()
top_traders.columns = ["account", "trade_count"]
top_traders["account"] = top_traders["account"].str[:12] + "…"
px.bar(top_traders, x="trade_count", y="account", orientation="h",
       title="Top 20 Accounts by Trade Count", template="plotly_dark").show()


## ✅ EDA Complete — proceed to `02_eda_sentiment.ipynb`